In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "DATA2.csv"   

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.astype(str).str.strip()

df.head()

,Date,Time,Temp_Out,Hi_Temp,Low_Temp,Out_Hum,Dew_Pt,Wind_Speed,Wind_Dir,Wind_Run,...,In_Hum,In_Dew,In_Heat,In_EMC,InAir_Density,ET,Wind_Samp,Wind_Tx,ISS_Recept,Arc_Int
0,13/11/2024,16:00,10.1,10.1,10.1,82,7.1,0,---,0,...,42,8.6,21.2,8.01,1.2070,0,0,1,0,30
1,13/11/2024,16:30,9.7,10.1,9.7,83,7,0,---,0,...,41,8.7,21.8,7.84,1.2048,0,0,4,0,30
2,13/11/2024,17:00,9.6,9.7,9.5,85,7.2,0,---,0,...,42,8.9,21.7,8.00,1.2052,0,0,4,0,30
3,13/11/2024,17:30,9.8,9.8,9.6,85,7.4,0,---,0,...,43,9.0,21.3,8.17,1.2060,0,0,4,0,30
4,13/11/2024,18:00,10.1,10.1,9.8,85,7.7,0,---,0,...,43,9.2,21.7,8.15,1.2047,0,0,4,0,30


In [ ]:

import pandas as pd
import numpy as np
import panel as pn
import hvplot.pandas  
import requests
from datetime import datetime

pn.extension('tabulator', notifications=True)

In [ ]:

# 1) CONFIG

CSV_PATH = "DATA2.csv"   

# Real-time weather location 
LAT = 53.8     
LON = -1.75

# Refresh frequency for live weather (seconds)
WEATHER_REFRESH_SECONDS = 300  # 5 minutes


In [ ]:
# Combine Date + Time 
date_str = df["Date"].astype(str).str.strip()
time_str = df["Time"].astype(str).str.strip()

df["datetime"] = pd.to_datetime(
    date_str + " " + time_str,
    dayfirst=True,
    errors="coerce"
)


if df["datetime"].isna().mean() > 0.8:
    df["datetime"] = pd.to_datetime(
        date_str + " " + time_str,
        dayfirst=False,
        errors="coerce"
    )


df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")


df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.to_period("M").astype(str)
df["weekday"] = df["datetime"].dt.day_name()

In [ ]:
# data quality report
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_%": (df.isna().mean() * 100).round(2),
    "min": df.select_dtypes(include=[np.number]).min(),
    "max": df.select_dtypes(include=[np.number]).max(),
}).sort_values(["missing_%", "dtype"], ascending=[False, True])

quality.head(15)


,dtype,missing_%,min,max
datetime,datetime64[ns],0.0,NaN,NaN
Bar,float64,0.0,966.8000,1045.6000
InAir_Density,float64,0.0,1.1124,1.2318
In_Dew,float64,0.0,-1.0000,15.8000
In_EMC,float64,0.0,4.6700,11.0500
In_Heat,float64,0.0,14.3000,32.9000
In_Temp,float64,0.0,15.4000,33.3000
Rain,float64,0.0,0.0000,10.2000
Rain_Rate,float64,0.0,0.0000,50.8000
hour,int32,0.0,0.0000,23.0000


Dashboard




In [ ]:

# Helpers for Real-time Weather

def fetch_live_weather(lat: float, lon: float) -> dict:
    """Fetch current weather from Open-Meteo (no key)."""
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        "&current=temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,pressure_msl"
        "&timezone=auto"
    )
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    j = r.json()
    cur = j.get("current", {})
    return {
        "time": cur.get("time"),
        "temperature_2m": cur.get("temperature_2m"),
        "relative_humidity_2m": cur.get("relative_humidity_2m"),
        "wind_speed_10m": cur.get("wind_speed_10m"),
        "wind_direction_10m": cur.get("wind_direction_10m"),
        "pressure_msl": cur.get("pressure_msl"),
    }

live_md = pn.pane.Markdown("Loading live weather…", sizing_mode="stretch_width")

def refresh_live_weather():
    try:
        w = fetch_live_weather(LAT, LON)
        live_md.object = f"""
### Live Weather (Open‑Meteo)

- **Time:** {w.get('time')}

- **Temperature (°C):** {w.get('temperature_2m')}

- **Humidity (%):** {w.get('relative_humidity_2m')}

- **Wind speed (km/h):** {w.get('wind_speed_10m')}

- **Wind direction (°):** {w.get('wind_direction_10m')}

- **Pressure (hPa):** {w.get('pressure_msl')}

_Location:_ lat={LAT}, lon={LON}
"""

    except Exception as e:
        live_md.object = f"""### Live Weather

⚠️ Could not fetch live data.

**Error:** `{e}`

Check internet access and LAT/LON values."""

refresh_live_weather()

# periodic refresh
cb = pn.state.add_periodic_callback(lambda: refresh_live_weather(), period=WEATHER_REFRESH_SECONDS*1000, start=True)

real_time_tab = pn.Column(
    pn.pane.Markdown("""## Real‑time Weather

This tab auto-refreshes every few minutes."""),
    live_md,
    pn.pane.Markdown("""Tip: If you're offline, this will show an error — the rest of the dashboard still works."""),
    sizing_mode="stretch_width"
)


In [ ]:

# Time Series Tab 

import numpy as np

# only numeric columns for plotting
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise ValueError("No numeric columns found for Time Series plots.")

metric = pn.widgets.Select(
    name="Metric",
    options=sorted(numeric_cols),
    value="Temp_Out" if "Temp_Out" in numeric_cols else sorted(numeric_cols)[0]
)

date_range = pn.widgets.DateRangeSlider(
    name="Date range",
    start=df["datetime"].min().date(),
    end=df["datetime"].max().date(),
    value=(df["datetime"].min().date(), df["datetime"].max().date())
)

resample = pn.widgets.Select(
    name="Resample",
    options=["None", "30min", "1H", "1D", "1W"],
    value="1H"
)

@pn.depends(metric, date_range, resample)
def ts_plot(metric, date_range, resample):
    start, end = date_range
    d = df[(df["datetime"].dt.date >= start) & (df["datetime"].dt.date <= end)].copy()

    if d.empty:
        return pn.pane.Markdown("No data in selected range.")

    d = d.set_index("datetime")[[metric]].dropna()

    if resample != "None":
        d = d.resample(resample).mean()

    return d.hvplot.line(height=360, width=900, title=f"{metric} over time")

time_series_tab = pn.Column(
    pn.pane.Markdown("## Time Series"),
    pn.Row(metric, resample),
    date_range,
    ts_plot,
    sizing_mode="stretch_width"
)

C:\Users\khang\AppData\Local\Temp\ipykernel_7764\55796405.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  d = d.resample(resample).mean()


In [ ]:

# Humidity & Temp Tab 

numeric_metrics = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_metrics:
    raise ValueError("No numeric columns found in df to plot.")

temp_options = [c for c in numeric_metrics if "temp" in c.lower()] or numeric_metrics
hum_options  = [c for c in numeric_metrics if "hum"  in c.lower()] or numeric_metrics

temp_col = pn.widgets.Select(
    name="Temperature column",
    options=temp_options,
    value="Temp_Out" if "Temp_Out" in temp_options else temp_options[0]
)

hum_col = pn.widgets.Select(
    name="Humidity column",
    options=hum_options,
    value="Out_Hum" if "Out_Hum" in hum_options else hum_options[0]
)

@pn.depends(temp_col, hum_col)
def scatter_th(tcol, hcol):
    d = df[[tcol, hcol]].apply(pd.to_numeric, errors="coerce").dropna()
    if d.empty:
        return pn.pane.Markdown("No data for this selection.")
    return d.hvplot.scatter(
        x=tcol, y=hcol,
        height=360, width=900,
        title=f"{hcol} vs {tcol}"
    )

@pn.depends(temp_col, hum_col)
def corr_card(tcol, hcol):
    d = df[[tcol, hcol]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(d) < 5:
        return pn.pane.Markdown("Not enough points for correlation.")
    corr = float(d.corr().iloc[0, 1])
    return pn.indicators.Number(
        name="Correlation",
        value=corr,
        format="{value:.3f}"
    )

hum_temp_tab = pn.Column(
    pn.pane.Markdown("## Humidity & Temperature"),
    pn.Row(temp_col, hum_col, corr_card),
    scatter_th,
    sizing_mode="stretch_width"
)

In [ ]:

# Wind Tab 

wind_speed_col = "Wind_Speed" if "Wind_Speed" in df.columns else None

dir_cols = [c for c in df.columns if c.lower().endswith("dir")]
wind_dir_col = "Wind_Dir" if "Wind_Dir" in df.columns else (dir_cols[0] if dir_cols else None)

def wind_layout():
    parts = []

    if wind_speed_col:
        parts.append(
            df[[wind_speed_col]].dropna()
            .hvplot.hist(bins=40, height=320, width=900, title="Wind speed distribution")
        )
    else:
        parts.append(pn.pane.Markdown("No Wind_Speed column found."))

    if wind_dir_col:
        top = (
            df[wind_dir_col].fillna("NA").astype(str)
            .value_counts().head(16)
            .rename_axis("direction")
            .reset_index(name="count")
        )
        parts.append(
            top.hvplot.bar(x="direction", y="count", height=320, width=900,
                           title="Wind direction counts (top 16)")
        )
    else:
        parts.append(pn.pane.Markdown("No wind direction column found."))

    return pn.Column(*parts, sizing_mode="stretch_width")

wind_tab = pn.Column(wind_layout(), sizing_mode="stretch_width")

In [ ]:

# Data Table Tab 
table_cols = df.columns.tolist()

table = pn.widgets.Tabulator(
    df[table_cols],
    pagination="remote",
    page_size=20,
    sizing_mode="stretch_width",
    height=520
)

data_table_tab = pn.Column(
    pn.pane.Markdown(
        "## Data Table (filter/sort)\nUse column filters (funnel icon) to filter."
    ),
    table,
    sizing_mode="stretch_width"
)


In [ ]:

# 4) Dashboard 

dashboard = pn.Tabs(
    ("Real‑time Weather", real_time_tab),
    ("Time Series", time_series_tab),
    ("Humidity & Temp", hum_temp_tab),
    ("Wind", wind_tab),
    ("Data Table", data_table_tab),
    dynamic=True
)

dashboard


Tabs(dynamic=True)
    [0] Column(sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] Markdown(str, sizing_mode='stretch_width')
        [2] Markdown(str)
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] Row
            [0] Select(name='Metric', options=['Arc_Int', 'Bar', ...], value='Arc_Int')
            [1] Select(name='Resample', options=['None', '30min', ...], value='1H')
        [2] DateRangeSlider(end=datetime.date(2025, ..., name='Date range', start=datetime.date(2024, ..., value=(datetime.date(2024, ..., value_end=datetime.date(2025, ..., value_start=datetime.date(2024, ...)
        [3] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [2] Column(sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] Row
            [0] Select(name='Temperature column', options=['In_Temp'], value='In_Temp')
            [1] Select(name='Humidity column', options=['In_Hum'], value='In_Hum')
            [2] ParamFunction(function, _pane=Number, defer_load=False)
        [2] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [3] Column(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] HoloViews(Histogram)
            [1] HoloViews(Bars)
    [4] Column(sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] Tabulator(height=520, page_size=20, pagination='remote', sizing_mode='stretch_width', value=             D...)